# 04 · Train Isolation Forest on the custom dataset

Reads `custom.csv` from the HF dataset repo, fits the shared
`CustomPreprocessor` from the project's `src/` package, trains an
`IsolationForest` in the classic unsupervised regime (legit-only training),
evaluates against a rule-based baseline (study research objective 4), and
pushes everything to the HF model repo.

In [ ]:
import sys, subprocess
for p in ('huggingface_hub','pandas','scikit-learn','joblib','statsmodels'):
    mod = {'scikit-learn':'sklearn'}.get(p, p)
    try: __import__(mod)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
print('deps OK')

In [ ]:
import sys
from pathlib import Path
APP_ROOT = Path('/home/user/app')
if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

import os, json
import numpy as np
import pandas as pd
import joblib
from huggingface_hub import HfApi, create_repo, hf_hub_download, whoami

from src.data.preprocessor import CustomPreprocessor
from src.models.rule_based_baseline import custom_rule_predict
from src.models.isolation_forest import if_predict_fraud

HF_TOKEN = os.environ['HF_TOKEN']
HF_USERNAME = os.environ.get('HF_USERNAME') or whoami(token=HF_TOKEN)['name']
DATASET_REPO = f'{HF_USERNAME}/fraud-detection-datasets'
MODEL_REPO   = f'{HF_USERNAME}/fraud-detection-models'

In [ ]:
csv = hf_hub_download(repo_id=DATASET_REPO, filename='custom.csv', repo_type='dataset', token=HF_TOKEN)
df = pd.read_csv(csv)
print(df.shape, 'fraud rate', df.Fraudulent.mean())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest

RANDOM_STATE = 42; TEST_SIZE = 0.2
pre = CustomPreprocessor()
X, y = pre.fit_transform(df)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)

contamination = float(max(0.001, min(0.5, y_tr.mean()))) if y_tr.sum() else 'auto'
iso = IsolationForest(
    n_estimators=150, max_samples='auto', contamination=contamination,
    random_state=RANDOM_STATE, n_jobs=-1,
)
iso.fit(X_tr[y_tr == 0])  # classic IF protocol: fit on legit-only majority
print('IF trained on', int((y_tr == 0).sum()), 'legit rows; contamination', contamination)

## Evaluate

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix)
from statsmodels.stats.contingency_tables import mcnemar

def evaluate(y_true, y_pred, y_score=None, label='model'):
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    tn, fp, fn, tp = cm.ravel()
    out = {'label': label,
           'accuracy': float(accuracy_score(y_true, y_pred)),
           'precision': float(precision_score(y_true, y_pred, zero_division=0)),
           'recall': float(recall_score(y_true, y_pred, zero_division=0)),
           'f1': float(f1_score(y_true, y_pred, zero_division=0)),
           'true_negatives': int(tn), 'false_positives': int(fp),
           'false_negatives': int(fn), 'true_positives': int(tp)}
    if y_score is not None and len(set(y_true)) > 1:
        out['roc_auc'] = float(roc_auc_score(y_true, y_score))
        fpr, tpr, _ = roc_curve(y_true, y_score)
        out['roc_curve'] = {'fpr': fpr.tolist(), 'tpr': tpr.tolist(), 'auc': out['roc_auc']}
    return out

y_pred, y_score = if_predict_fraud(iso, X_te)
iso_metrics = evaluate(y_te, y_pred, y_score, 'isolation_forest')

df_idx = df.reset_index(drop=True)
_, idx_te = train_test_split(df_idx.index, test_size=TEST_SIZE, stratify=df.Fraudulent, random_state=RANDOM_STATE)
rule_pred = custom_rule_predict(df_idx.loc[idx_te])
rule_metrics = evaluate(y_te, rule_pred, None, 'rule_based_custom')

iso_right = (y_pred == y_te); base_right = (rule_pred == y_te)
a = int((iso_right & base_right).sum()); b = int((base_right & ~iso_right).sum())
c = int((~base_right & iso_right).sum()); d = int((~iso_right & ~base_right).sum())
mc = mcnemar([[a,b],[c,d]], exact=(b+c)<25, correction=True)
mcnemar_result = {
    'table': {'ml_right_base_right': a,'ml_wrong_base_right': b,'ml_right_base_wrong': c,'ml_wrong_base_wrong': d},
    'statistic': float(mc.statistic) if mc.statistic is not None else None,
    'p_value': float(mc.pvalue),
    'reject_null_at_0.05': bool(mc.pvalue < 0.05),
    'interpretation': ('IF significantly improves accuracy (reject H1, accept HA)'
                       if mc.pvalue < 0.05 and c > b else 'Insufficient evidence to reject H1'),
}
print(json.dumps({'iso': iso_metrics,'rule': rule_metrics,'mcnemar': mcnemar_result}, indent=2)[:1200])

## Persist + push

In [ ]:
OUT = Path('/tmp/fraudguard_artifacts'); OUT.mkdir(exist_ok=True)
joblib.dump(iso, OUT / 'isolation_forest.joblib')
joblib.dump(pre, OUT / 'custom_preprocessor.joblib')

create_repo(repo_id=MODEL_REPO, repo_type='model', token=HF_TOKEN, private=True, exist_ok=True)
existing = {}
try:
    existing = json.loads(Path(hf_hub_download(repo_id=MODEL_REPO, filename='metadata.json', token=HF_TOKEN)).read_text())
except Exception:
    pass
existing['custom'] = {
    'isolation_forest': iso_metrics,
    'rule_based': rule_metrics,
    'mcnemar': mcnemar_result,
    'contamination': contamination if isinstance(contamination, float) else None,
    'n_train_legit_only': int((y_tr == 0).sum()),
}
(OUT / 'metadata.json').write_text(json.dumps(existing, indent=2, default=str))

api = HfApi()
for f in ('isolation_forest.joblib','custom_preprocessor.joblib','metadata.json'):
    print('upload', f)
    api.upload_file(path_or_fileobj=str(OUT / f), path_in_repo=f,
                    repo_id=MODEL_REPO, repo_type='model', token=HF_TOKEN,
                    commit_message=f'Update {f} from IF training')
print('done — view at https://huggingface.co/' + MODEL_REPO)